# RAG systems

Simple RAG system

We will use:
- embedding model hf.co/CompendiumLabs/bge-base-en-v1.5-gguf
- Language model: hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF
- "database" - just a simple text file with some information (phrases) 
We will create a simple RAG system that retrieves relevant information from the text file based on the user's query and then generates a response using the language model.




## Prerequisites 

let's start by installing ollama from project's website: ollama.com

After installed, open a terminal and run the following command to download the required models:

ollama pull hf.co/CompendiumLabs/bge-base-en-v1.5-gguf
ollama pull hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF

install the ollama package

In [ ]:
pip install ollama

## Indexing phase

### select a dataset

In [1]:
# loading the dataset
dataset = []
with open('cat-facts.txt', 'r', encoding='utf-8') as file:
    dataset = file.readlines()
    print(f'Loaded {len(dataset)} entries')

Loaded 150 entries


### select another dataset

In [ ]:
# pip install datasets

In [ ]:
# from datasets import load_dataset

# # check all the available datasets
# from huggingface_hub import list_datasets
# print([dataset.id for dataset in list_datasets(limit=100)])

# # choose a dataset and load it
# dataset = load_dataset("open-index/hacker-news")
# print(dataset["train"])

['OpenMOSS-Team/OmniAction', 'open-index/hacker-news', 'th1nhng0/vietnamese-legal-documents', 'OpenMOSS-Team/OmniAction-LIBERO', 'ServiceNow-AI/eva', 'ropedia-ai/xperience-10m', 'nohurry/Opus-4.6-Reasoning-3000x-filtered', 'Roman1111111/claude-opus-4.6-10000x', 'OpenSpeechHub/Genshin-Voice-Ja', 'internlm/WildClawBench', 'nvidia/Nemotron-Cascade-2-RL-data', 'OpenSQZ/AutoMathText-V2', 'stepfun-ai/Step-3.5-Flash-SFT', 'Hafftka/michael-hafftka-catalog-raisonne', 'nvidia/Nemotron-Cascade-2-SFT-Data', 'purvanshi/lica-data', 'ibm-research/VAKRA', 'zai-org/ZClawBench', 'mercor/APEX-SWE', 'ServiceNow-AI/EnterpriseOps-Gym', 'bones-studio/seed', 'Jackrong/Qwen3.5-reasoning-700x', 'markov-ai/computer-use-large', 'GaryYang123/zh-meme-sft-8k', 'ServiceNow/VideoCUA', 'nvidia/Nemotron-Personas-France', 'ytu-ce-cosmos/tubitak-science-olympiad-tr', 'openai/gsm8k', 'tencent/Penguin-Recap-I', 'Idavidrein/gpqa', 'Anthropic/EconomicIndex', 'allenai/olmOCR-bench', 'Crownelius/Opus-4.6-Reasoning-3300x', 'Teic

### compute the embeddings and build the index

In [ ]:
# vectorize the dataset
import ollama

EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF:latest'

# Each element in the VECTOR_DB will be a tuple (chunk, embedding)
# The embedding is a list of floats, for example: [0.1, 0.04, -0.34, 0.21, ...]
VECTOR_DB = []

def add_chunk_to_database(chunk):
    embedding = ollama.embed(model=EMBEDDING_MODEL, input=chunk)['embeddings'][0]
    VECTOR_DB.append((chunk, embedding))

# one line = one chunk in this example, but you can use more complex chunking strategies
for i, chunk in enumerate(dataset):
    add_chunk_to_database(chunk)
    print(f'Added chunk {i+1}/{len(dataset)} to the database')

In [3]:
# retrieval function using cosine similarity

def cosine_similarity(a, b):
    dot_product = sum([x * y for x, y in zip(a, b)])
    norm_a = sum([x ** 2 for x in a]) ** 0.5
    norm_b = sum([x ** 2 for x in b]) ** 0.5
    return dot_product / (norm_a * norm_b)

def retrieve(query, top_n=3):
    query_embedding = ollama.embed(model=EMBEDDING_MODEL, input=query)['embeddings'][0]
    # temporary list to store (chunk, similarity) pairs
    similarities = []
    for chunk, embedding in VECTOR_DB:
        similarity = cosine_similarity(query_embedding, embedding)
        similarities.append((chunk, similarity))
    # sort by similarity in descending order, because higher similarity means more relevant chunks
    similarities.sort(key=lambda x: x[1], reverse=True)
    # finally, return the top N most relevant chunks
    return similarities[:top_n]

## Generation phase  - chatbot interaction

### query the system

In [5]:
# build a prompt
input_query = input('Ask me a question: ')
retrieved_knowledge = retrieve(input_query)

print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
    print(f' - (similarity: {similarity:.2f}) {chunk}')



Retrieved knowledge:
 - (similarity: 0.70) A cat cannot see directly under its nose. This is why the cat cannot seem to find tidbits on the floor.

 - (similarity: 0.69) The ability of a cat to find its way home is called “psi-traveling.” Experts think cats either use the angle of the sunlight to find their way or that cats have magnetized cells in their brains that act as compasses.

 - (similarity: 0.67) A group of cats is called a “clowder.”



### generate the response

In [7]:
# use the LLM to generate the response
context = '\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])
instruction_prompt = f'''You are a helpful chatbot. Use only the following pieces of context to answer the question. \n
                        Don't make up any new information: {context}'''

stream = ollama.chat(
    model=LANGUAGE_MODEL,
    messages=[
    {'role': 'system', 'content': instruction_prompt},
    {'role': 'user', 'content': input_query},
    ],
    stream=True,
)

# print the response from the chatbot in real-time
print('Chatbot response:')
for chunk in stream:
    print(chunk['message']['content'], end='', flush=True)


Chatbot response:
You can't see it, but I think your cat might be hiding under its bed. It could be using the sunlight to help guide it towards home, like "psi-traveling." Maybe you have a cozy little ball of fur in there with you!

# 